In [0]:
import pandas as pd
import requests
import joblib
import mlflow.xgboost
import math
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

In [0]:
# ==========================================
# 1. USER INPUTS (Change these to test!)
# ==========================================
FLIGHT_STRING = "DL1246"             
ORIGIN = "LIT"                       # Explicit 3-letter Origin code
DEST = "ATL"                         # Explicit 3-letter Destination code
DEPARTURE_DATE = "2026-04-02"        # YYYY-MM-DD
DEPARTURE_TIME_LOCAL = "14:30"       # HH:MM (24-hour format)

In [0]:
# ==========================================
# 2. CONFIGURATION & MAPPING
# ==========================================
AIRPORT_COORDS = {
    "LIT": {"lat": 34.7294, "lon": -92.2243}, "DFW": {"lat": 32.8998, "lon": -97.0403},
    "ATL": {"lat": 33.6407, "lon": -84.4277}, "ORD": {"lat": 41.9742, "lon": -87.9073},
    "DEN": {"lat": 39.8561, "lon": -104.6737}, "LAS": {"lat": 36.0840, "lon": -115.1537},
    "CLT": {"lat": 35.2140, "lon": -80.9431}, "MIA": {"lat": 25.7959, "lon": -80.2870},
    "LGA": {"lat": 40.7769, "lon": -73.8740}, "DCA": {"lat": 38.8512, "lon": -77.0402},
    "IAH": {"lat": 29.9902, "lon": -95.3368}, "DAL": {"lat": 32.8471, "lon": -96.8517},
    "STL": {"lat": 38.7499, "lon": -90.3748}
}

def map_weather_code(code):
    """Maps Open-Meteo WMO codes to our categorical text strings"""
    if code in [0, 1]: return "Clear"
    if code in [2, 3, 45, 48]: return "Cloudy"
    if 50 <= code <= 69 or 80 <= code <= 82: return "Rain"
    if 70 <= code <= 79 or 85 <= code <= 86: return "Snow"
    if code >= 95: return "Storm"
    return "Clear"

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates geodetic distance in miles if history is missing."""
    R = 3958.8 # Radius of Earth in miles
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [0]:
# ==========================================
# 3. DATE/TIME & COORDINATE GUARDRAILS
# ==========================================
print("1. Validating Inputs...")
target_dt = datetime.strptime(f"{DEPARTURE_DATE} {DEPARTURE_TIME_LOCAL}", "%Y-%m-%d %H:%M")
now = datetime.now()

if target_dt < now:
    raise ValueError(f"ERROR: The date {target_dt} is in the past. This model predicts future flights.")
if target_dt > now + timedelta(days=14):
    raise ValueError(f"ERROR: The date {target_dt} is too far in the future. Weather APIs only forecast 14 days out.")
if ORIGIN not in AIRPORT_COORDS or DEST not in AIRPORT_COORDS:
    raise ValueError(f"ERROR: Origin ({ORIGIN}) or Dest ({DEST}) coordinates not mapped. Please add them to AIRPORT_COORDS.")

In [0]:
# ==========================================
# 4. DATABASE / DISTANCE LOOKUP
# ==========================================
print(f"2. Determining Route Distance for {ORIGIN} -> {DEST}...")
airline_code = ''.join([c for c in FLIGHT_STRING if c.isalpha()])

silver_flights = spark.table("aviation_project.silver_historical_flights")
route_info = silver_flights.filter(
    (F.col("origin_airport") == ORIGIN) & 
    (F.col("destination_airport") == DEST)
).select("distance_miles").first()

if route_info and route_info['distance_miles']:
    distance = float(route_info['distance_miles'])
    print(f"   -> Historical distance found: {distance:.1f} miles")
else:
    print("   -> Route not found in history. Calculating geodetic distance...")
    distance = haversine_distance(
        AIRPORT_COORDS[ORIGIN]['lat'], AIRPORT_COORDS[ORIGIN]['lon'],
        AIRPORT_COORDS[DEST]['lat'], AIRPORT_COORDS[DEST]['lon']
    )
    print(f"   -> Calculated distance: {distance:.1f} miles")

In [0]:
# ==========================================
# 5. WEATHER FORECAST API INTEGRATION
# ==========================================
print("3. Fetching Live Weather Forecasts from Open-Meteo...")

def get_forecast(airport_code, target_datetime):
    coords = AIRPORT_COORDS[airport_code]
    url = f"https://api.open-meteo.com/v1/forecast?latitude={coords['lat']}&longitude={coords['lon']}&hourly=temperature_2m,precipitation,wind_speed_10m,weathercode&temperature_unit=fahrenheit&wind_speed_unit=mph&precipitation_unit=inch&timezone=auto&forecast_days=14"
    
    response = requests.get(url).json()
    target_str = target_datetime.strftime("%Y-%m-%dT%H:00")
    
    try:
        idx = response['hourly']['time'].index(target_str)
        return {
            "temp": response['hourly']['temperature_2m'][idx],
            "wind": response['hourly']['wind_speed_10m'][idx],
            "precip": response['hourly']['precipitation'][idx],
            "condition": map_weather_code(response['hourly']['weathercode'][idx])
        }
    except ValueError:
        raise ValueError(f"Could not align {target_str} with the weather forecast timeline.")

origin_weather = get_forecast(ORIGIN, target_dt)
dest_weather = get_forecast(DEST, target_dt)

print(f"   -> {ORIGIN} Forecast: {origin_weather['temp']}F, {origin_weather['wind']}mph, {origin_weather['condition']}")
print(f"   -> {DEST} Forecast: {dest_weather['temp']}F, {dest_weather['wind']}mph, {dest_weather['condition']}")

In [0]:
# ==========================================
# 6. FEATURE ENGINEERING
# ==========================================
print("4. Engineering Final ML Matrix...")
minute_of_day = (target_dt.hour * 60) + target_dt.minute
is_holiday = 1 if (target_dt.month == 12 and target_dt.day >= 20) or \
                  (target_dt.month == 1 and target_dt.day <= 4) or \
                  (target_dt.month == 11 and 20 <= target_dt.day <= 29) or \
                  (target_dt.month == 7 and 3 <= target_dt.day <= 6) else 0

my_flight = pd.DataFrame([{
    'distance_miles': distance,
    'origin_temp': origin_weather['temp'],
    'origin_wind': origin_weather['wind'],
    'origin_precip': origin_weather['precip'],
    'dest_temp': dest_weather['temp'],
    'dest_wind': dest_weather['wind'],
    'dest_precip': dest_weather['precip'],
    'time_sin': math.sin(minute_of_day * (2 * math.pi / 1440)),
    'time_cos': math.cos(minute_of_day * (2 * math.pi / 1440)),
    'is_holiday': is_holiday,
    'month': target_dt.month,
    'day_of_month': target_dt.day,
    'day_of_week': target_dt.weekday() + 1,
    'airline_code': airline_code,
    'origin_condition': origin_weather['condition'],
    'dest_condition': dest_weather['condition']
}])

encoder = joblib.load('/Volumes/workspace/aviation_project/raw_data/ordinal_encoder.pkl')
categorical_cols = ['airline_code', 'origin_condition', 'dest_condition']
my_flight[categorical_cols] = encoder.transform(my_flight[categorical_cols])

In [0]:
# ==========================================
# 7. INFERENCE ENGINE (Prediction)
# ==========================================
print("5. Loading MLflow Model and Predicting...\n")

# *** YOU MUST UPDATE THIS RUN ID TO MATCH YOUR XGBOOST RUN ***
RUN_ID = "runs:/7d54c61ff67149c4b2def551290278b6/xgboost-model" 
loaded_model = mlflow.xgboost.load_model(RUN_ID)

probabilities = loaded_model.predict_proba(my_flight)[0]

print("="*40)
print(f" FLIGHT FORECAST: {FLIGHT_STRING} ({ORIGIN} -> {DEST})")
print(f" DATE: {DEPARTURE_DATE} | TIME: {DEPARTURE_TIME_LOCAL}")
print("="*40)
print(f" ✅ On Time:           {probabilities[0] * 100:.2f}%")
print(f" ⚠️ Delayed:           {probabilities[1] * 100:.2f}%")
print(f" 🚨 Canceled/Diverted: {probabilities[2] * 100:.2f}%")
print("="*40)